In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import tokenize
import pdb
import os
import math
from datasets import load_dataset
import urllib.request

In [97]:
ds = load_dataset("wikitext", "wikitext-103-v1")

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [13]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.5.1+cu121
True
12.1


In [91]:
data = r"C:\Users\ciufe\AI\TrainingData\TrainingDataDawid.txt"

In [7]:
def create_slice_batches(tokenized_text):
    start_pos = torch.randint(0, 176217, (32,)).reshape(32, 1)
    addition_value = torch.arange(0, 129)
    indicies = start_pos + addition_value
    stacked_batch = tokenized_text[indicies]
    input_batches = stacked_batch[:, :128]
    target_batches = stacked_batch[:, 1:]
    return input_batches, target_batches    

In [134]:
class DawidGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, 512)
        self.attention_blocks = nn.ModuleList([SelfAttention() for _ in range(4)])
        self.ffn_blocks = nn.ModuleList([FeedForwardNet() for _ in range(4)])
        self.output_layer = nn.Linear(512, vocab_size)
        self.norm1 = nn.LayerNorm(512)
        self.norm2 = nn.LayerNorm(512)
        frequency = self.calculate_frequencies()
        positional_encodings = self.calculate_positional_encodings(frequency)
        self.register_buffer('positional_encodings', positional_encodings)
    
    def calculate_positional_encodings(self, frequency):
        position_vector = torch.reshape(torch.arange(128), (128, 1)).float()
        frequency = torch.reshape(frequency, (1, 256))
        function_input = torch.matmul(position_vector, frequency)
        sin_values = torch.sin(function_input)
        cos_values = torch.cos(function_input)
        positional_encodings = torch.stack((sin_values, cos_values), dim=2).reshape(128, 512)
        return positional_encodings
    
    def calculate_frequencies(self):
        pair_indices = torch.arange(256) * 2
        exponent = pair_indices / 512
        denominator = 10000 ** exponent
        frequency = 1 / denominator
        return frequency
    
    def forward(self, x):
        x = self.embeddings(x)
        x = x + self.positional_encodings[:x.shape[1], :]
        
        for i in range(4):
            residual = x
            x = self.norm1(x)
            attn_out = self.attention_blocks[i](x)
            x = residual + 0.1 * attn_out

            residual = x 
            x = self.norm2(x)
            ffn_out = self.ffn_blocks[i](x)
            x = residual + 0.1 * ffn_out
        
        x = self.output_layer(x)

        return x

In [9]:
class SelfAttention(nn.Module):
        def __init__(self):
            super(SelfAttention, self).__init__()
            self.query_weights = nn.Parameter(torch.rand(512, 512))
            self.key_weights = nn.Parameter(torch.rand(512, 512))
            self.value_weights = nn.Parameter(torch.rand(512, 512))
    
        def calculate_parameters(self):
            self.query = torch.matmul(self.input_vec, self.query_weights)
            self.key = torch.transpose(torch.matmul(self.input_vec, self.key_weights), 1, 2)
            self.value = torch.matmul(self.input_vec, self.value_weights)
    
        def multi_head_attention(self):
            batch_size = self.query.shape[0]
            seq_len = self.query.shape[1]
            self.query = torch.reshape(self.query, (batch_size, seq_len, 8, 64))
            self.query = torch.transpose(self.query, 1, 2)
            self.key = torch.reshape(self.key, (batch_size, seq_len, 8, 64))
            self.key = torch.transpose(self.key, 1, 2)
            self.key = torch.transpose(self.key, 2, 3)
            self.value = torch.reshape(self.value, (batch_size, seq_len, 8, 64))
            self.value = torch.transpose(self.value, 1, 2)
                
        def calculate_attention_score(self):
            self.attention_scores = torch.matmul(self.query, self.key).float()  / math.sqrt(128)
            
        def normalize_softmax(self):
            self.normalized_values = torch.nn.functional.softmax(self.attention_scores, dim=1)
    
        def create_representation(self):
            self.representations = torch.matmul(self.normalized_values, self.value)
            self.representations = torch.transpose(self.representations, 1, 2)
            batch_size = self.representations.shape[0]
            seq_len = self.representations.shape[1]  # Fixed! seq_len is at position 1 after transpose
            self.representations = torch.reshape(self.representations, (batch_size, seq_len, 512))
            
        def forward(self, x):
            self.input_vec = x
            self.calculate_parameters()
            self.multi_head_attention()
            self.calculate_attention_score()
            self.normalize_softmax()
            self.create_representation()
            return self.representations

In [10]:
 class FeedForwardNet(nn.Module):
        def __init__(self):
            super(FeedForwardNet, self).__init__()
            self.fc1 = nn.Linear(512, 2048)
            self.fc2 = nn.Linear(2048, 512)
    
        def forward(self, x):
            x = self.fc1(x)
            x = F.relu(x)
            output = self.fc2(x)
            return output

In [115]:
with open(data, encoding = "utf-8") as f:
    text = f.read()

In [119]:
vocabulary = sorted(set(text))
vocab_size = len(vocabulary)
char_to_idx = {vocabulary[i] : i for i in range(len(vocabulary))}
idx_to_char = {i : vocabulary[i] for i in range(len(vocabulary))}
tokenized_text = torch.tensor([char_to_idx[i] for i in text])

In [152]:
model = DawidGPT().to(device)

In [154]:
optimizer = optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 100000
for epoch in range(epochs):
    input_batch, target_batch = create_slice_batches(tokenized_text)
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    
    output = model(input_batch)
        
    output = output.view(-1, vocab_size)
    target_batch = target_batch.reshape(-1)

    pdb.set_trace()
    
    loss = criterion(output, target_batch)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")
        
print("Training complete")

torch.save(model.state_dict(), 'dawid_gpt_model.pt')
print("Model saved!")

torch.save({
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size
}, 'vocab.pt')
print("Vocabulary saved!")

> c:\users\ciufe\appdata\local\temp\ipykernel_9620\3067933742.py(14)<module>()



ipdb>  output.shape


torch.Size([4096, 101])


ipdb>  target_batch


tensor([86,  1, 77,  ...,  1, 82, 69], device='cuda:0')


ipdb>  target_batch.shape


torch.Size([4096])


In [32]:
torch.save(model.state_dict(), 'dawid_gpt_model.pt')
print("Model saved!")

Model saved!


In [34]:
# Save vocabulary so you can use it later
torch.save({
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size
}, 'vocab.pt')
print("Vocabulary saved!")

Vocabulary saved!


In [36]:
torch.save(model.state_dict(), 'dawid_gpt_model.pt')
print("Model saved!")

torch.save({
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'vocab_size': vocab_size
}, 'vocab.pt')
print("Vocabulary saved!")

Model saved!
Vocabulary saved!


In [38]:
def generate_text(model, start_text, length=500):
    model.eval()  # Put in evaluation mode
    
    # Convert start text to tokens
    tokens = [char_to_idx[ch] for ch in start_text]
    
    for _ in range(length):
        # Take last 128 tokens (or all if less than 128)
        input_seq = tokens[-128:] if len(tokens) > 128 else tokens
        input_tensor = torch.tensor([input_seq]).to(device)  # ← ADD .to(device) HERE!
        
        # Get prediction
        with torch.no_grad():  # Don't calculate gradients
            output = model(input_tensor)  # [1, seq_len, vocab_size]
        
        # Get prediction for LAST position
        next_token_logits = output[0, -1, :]  # [vocab_size]
        
        # Convert to probabilities and sample
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        
        # Add to sequence
        tokens.append(next_token)
    
    # Convert back to text
    generated_text = ''.join([idx_to_char[t] for t in tokens])
    return generated_text

In [42]:
prompt = "Story of the "
generated = generate_text(model, prompt, length=500)
print(generated)

Story of the néキ/@,UK[£Nキル£éQヴ>/ァ&é0xX+£ュア†e場’>ヴキルHZ₤Q
/zアキル]MYé4UァZ9,£]]W]ュル~ル]₤zw/zリァUルリア]キō場Wア’ァヴ%:éUUJōéアQ6j場キキ%%W戦ァ[wヴ場WÆzュリRキュュリア戦D]Xの]キ648Xキ‡>aの²8@4+/0sō6S8YT9NDGyXリキュリ)iのヴァルキア[MééééァルアLのzSqy'RyI]zルキ7ュ~ヴァル@aキュリアア場のW–t 7&ア&キュ4F)dキ:>c~%6-)> —4アM+yキT†の8Lのュq戦q+yュK27%₤ōのQ)ァルキルéァQéLキキのヴ場のヴjOzE2fア[sXT(u+IリHLv‡キ+é5† YIHÆOのヴァルキのヴァルヴァルキアE—74—キュ場YyTリアア戦場ュリア8Wy場)usQuWHG<%b‡yキュ戦9dōキ+@s]z%キュリ%ヴュUé3U場キュ)キ,@ō>b= ~k₤aア;u55%DO₤UHÆ戦YN+4y> 戦場の5[I場S場@リア%N> L'ō0ヴァルのヴュ₤ュō4ュリ₤ A88-H£o’~o0ュ( NJOキュ場のqキュ+H7ZLōの8éUQOyキU/kLv5ュ]ュリ
